# 05 - Orchestration: the Airflow DAG (Rubric item 4 / 15 pts)

**Project:** ShopSense | **Program:** SDAIA Academy - Modern Data Engineering for AI Systems

## What this notebook must prove
| Rubric requirement | Where it is proven |
|---|---|
| A **real Airflow DAG**, not a custom orchestrator class | Section 3 - `apache-airflow 3.1.8`, `dags/shopsense_pipeline.py` |
| It **wires every stage together** | ingest -> gate -> silver -> gate -> (gold ‖ rag) -> complete |
| **Correct task dependencies** | Section 4 - the dependency graph printed from the parsed DAG |
| **A failed quality gate halts the pipeline before downstream stages run** | Section 6 - a real run where the gate fails and every downstream task ends `upstream_failed` |

The DAG imports `src/transforms.py`, `src/quality.py` and `src/lineage.py` written by notebook 04,
so the orchestrated pipeline and the gate are the *same code*, not a reimplementation.

> **Airflow 2.x does not support Python 3.13, which is what Colab runs.** This notebook therefore
> uses Airflow 3.1.8 with the official constraint file for 3.13.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# AIRFLOW_HOME must be set BEFORE airflow is imported anywhere in this session.
import os, sys
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/sdaia_capstone')
AIRFLOW_HOME = Path('/content/airflow')
DAGS_DIR     = AIRFLOW_HOME / 'dags'
DAGS_DIR.mkdir(parents=True, exist_ok=True)

os.environ['AIRFLOW_HOME']              = str(AIRFLOW_HOME)
os.environ['AIRFLOW__CORE__DAGS_FOLDER']= str(DAGS_DIR)
os.environ['AIRFLOW__CORE__LOAD_EXAMPLES'] = 'False'
os.environ['SHOPSENSE_PROJECT']         = str(PROJECT)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

for f in ['src/transforms.py', 'src/quality.py', 'src/lineage.py']:
    assert (PROJECT / f).exists(), f'{f} missing - run notebook 04 first'
print('AIRFLOW_HOME :', AIRFLOW_HOME)
print('DAGs folder  :', DAGS_DIR)
print('src modules  : found')

AIRFLOW_HOME : /content/airflow
DAGs folder  : /content/airflow/dags
src modules  : found


### Install Airflow

This is the slowest cell in the whole capstone - roughly **4 to 6 minutes** and a few hundred MB.
The constraint file is what keeps the install reproducible; without it pip resolves a broken set.

In [3]:
AIRFLOW_VERSION = '3.1.8'
PY = f'{sys.version_info.major}.{sys.version_info.minor}'
CONSTRAINTS = (f'https://raw.githubusercontent.com/apache/airflow/'
               f'constraints-{AIRFLOW_VERSION}/constraints-{PY}.txt')
print('python', PY, '\nconstraints:', CONSTRAINTS)

!pip install -q "apache-airflow=={AIRFLOW_VERSION}" --constraint "{CONSTRAINTS}"
!pip install -q "great-expectations==1.22.0" "openlineage-python==1.53.0" "deltalake>=1.0"

python 3.13 
constraints: https://raw.githubusercontent.com/apache/airflow/constraints-3.1.8/constraints-3.13.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.7/305.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.4/150.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [4]:
import airflow, great_expectations as gx, openlineage, deltalake
print('airflow           ', airflow.__version__)
print('great_expectations', gx.__version__)
print('deltalake         ', deltalake.__version__)

2026-09-09T16:21:23.373354Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:21:23.375929Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:21:23.379708Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:21:23.385343Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:21:23.393555Z [info     ] setup plugin alembic.autogenerate.defaults [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:21:23.399821Z [info     ] setup plugin alembic.autogenerate.comments [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:21:24.520636Z [info     ] NumExpr defaulting to 2 threads. [numexpr.utils] loc=utils.py:164
2026-09-09T16:21:33.315550Z [info     ] Skipping registering function get_context because it does not have a class [great_expectations.

airflow            3.1.8
great_expectations 1.22.0
deltalake          1.6.3


## 2. Initialise the Airflow metadata database

In [5]:
import subprocess

def sh(cmd, tail=6, quiet=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=os.environ)
    out = (r.stdout + r.stderr).strip().splitlines()
    if not quiet:
        print('\n'.join(out[-tail:]) if out else '(no output)')
    return r

sh('airflow db migrate', tail=3)
print('\nairflow version:', sh('airflow version', quiet=True).stdout.strip().splitlines()[-1])

2026-09-09T16:22:08.082147Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:213
2026-09-09T16:22:08.169235Z [info     ] Running stamp_revision  -> 509b94a1042d [alembic.runtime.migration] loc=migration.py:621
2026-09-09T16:22:08.185292Z [info     ] Airflow database tables created [airflow.utils.db] loc=db.py:709

airflow version: 3.1.8


## 3. The DAG

Six tasks. Two of them are quality gates that call `quality.gate()`, which **raises** on bad data -
and a raised exception is exactly what makes Airflow stop the branch.

```
ingest_orders → quality_gate_bronze → build_silver → quality_gate_silver ─┬→ build_gold ────────┬→ pipeline_complete
                                                                          └→ refresh_rag_index ─┘
```

`retries=0` so the failure demonstration finishes quickly. Every task body is wrapped in
`lineage.stage(...)`, which emits START, then COMPLETE or FAIL.

In [6]:
%%writefile /content/airflow/dags/shopsense_pipeline.py
# ShopSense end-to-end pipeline.
# Ingestion -> Bronze -> quality gate -> Silver -> quality gate -> Gold + RAG refresh.
# A failing gate raises, so Airflow marks every downstream task upstream_failed.
import os
import sys

import pendulum
from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator

PROJECT = os.environ.get('SHOPSENSE_PROJECT', '/content/drive/MyDrive/sdaia_capstone')
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from src import transforms, quality, lineage      # noqa: E402

DS_KAFKA  = 'kafka.orders.valid'
DS_BRONZE = 'lakehouse_dag.bronze.orders'
DS_SILVER = 'lakehouse_dag.silver.orders'
DS_GOLD   = 'lakehouse_dag.gold.daily_category_revenue'
DS_INDEX  = 'rag.knowledge_base_index'


def ingest_orders(**_):
    override = os.environ.get('SHOPSENSE_LANDING_OVERRIDE') or None
    with lineage.stage('ingest_orders', inputs=[DS_KAFKA], outputs=[DS_BRONZE]):
        info = transforms.ingest_to_bronze(override)
    print('ingested:', info)
    return info


def quality_gate_bronze(**_):
    with lineage.stage('quality_gate_bronze', inputs=[DS_BRONZE]):
        summary = quality.gate(transforms.read_bronze(), 'bronze')
    print(f"bronze gate PASSED: {summary['passed']}/{summary['evaluated']} expectations "
          f"over {summary['rows']} rows")
    return {k: summary[k] for k in ('rows', 'evaluated', 'passed', 'failed')}


def build_silver(**_):
    with lineage.stage('build_silver', inputs=[DS_BRONZE], outputs=[DS_SILVER]):
        info = transforms.build_silver()
    print('silver:', info)
    return info


def quality_gate_silver(**_):
    with lineage.stage('quality_gate_silver', inputs=[DS_SILVER]):
        summary = quality.gate(transforms.read_silver(), 'silver')
    print(f"silver gate PASSED: {summary['passed']}/{summary['evaluated']} expectations "
          f"over {summary['rows']} rows")
    return {k: summary[k] for k in ('rows', 'evaluated', 'passed', 'failed')}


def build_gold(**_):
    with lineage.stage('build_gold', inputs=[DS_SILVER], outputs=[DS_GOLD]):
        info = transforms.build_gold()
    print('gold:', info)
    return info


def refresh_rag_index(**_):
    with lineage.stage('refresh_rag_index', outputs=[DS_INDEX]):
        info = transforms.refresh_rag_index()
    print('rag index:', info)
    return info


def pipeline_complete(**_):
    with lineage.stage('pipeline_complete', inputs=[DS_GOLD, DS_INDEX]):
        print('all stages completed and both quality gates passed')
    return 'ok'


with DAG(
    dag_id='shopsense_pipeline',
    description='Kafka -> Delta medallion -> RAG refresh, gated by Great Expectations',
    start_date=pendulum.datetime(2026, 9, 1, tz='UTC'),
    schedule='@daily',
    catchup=False,
    max_active_runs=1,
    default_args={'owner': 'shopsense', 'retries': 0},
    tags=['sdaia', 'capstone', 'medallion', 'rag'],
) as dag:

    t_ingest = PythonOperator(task_id='ingest_orders',       python_callable=ingest_orders)
    t_gate_b = PythonOperator(task_id='quality_gate_bronze', python_callable=quality_gate_bronze)
    t_silver = PythonOperator(task_id='build_silver',        python_callable=build_silver)
    t_gate_s = PythonOperator(task_id='quality_gate_silver', python_callable=quality_gate_silver)
    t_gold   = PythonOperator(task_id='build_gold',          python_callable=build_gold)
    t_rag    = PythonOperator(task_id='refresh_rag_index',   python_callable=refresh_rag_index)
    t_done   = PythonOperator(task_id='pipeline_complete',   python_callable=pipeline_complete)

    t_ingest >> t_gate_b >> t_silver >> t_gate_s >> [t_gold, t_rag] >> t_done

Writing /content/airflow/dags/shopsense_pipeline.py


## 4. Register the DAG and show its dependency graph

In [7]:
sh('airflow dags reserialize', tail=2)
print()
sh('airflow dags list', tail=6)

2026-09-09T16:22:36.597425Z [info     ] Sync 1 DAGs                    [airflow.serialization.serialized_objects] loc=serialized_objects.py:2949
2026-09-09T16:22:36.650502Z [info     ] Setting next_dagrun for shopsense_pipeline to 2026-09-09 00:00:00+00:00, run_after=2026-09-09 00:00:00+00:00 [airflow.models.dag] loc=dag.py:693

2026-09-09T16:22:41.837887Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:22:41.838263Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:22:41.838506Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:22:41.838691Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:22:41.838855Z [info     ] setup plugin alembic.autogenerate.defaults [alembic.runtime.plugins] loc=plugins.py:37
2026-09-09T16:22:41.839031Z [info    

CompletedProcess(args='airflow dags list', returncode=0, stdout='dag_id             | fileloc                                     | owners    | is_paused | bundle_name | bundle_version\n===================+=============================================+===========+===========+=============+===============\nshopsense_pipeline | /content/airflow/dags/shopsense_pipeline.py | shopsense | True      | dags-folder | None          \n                                                                                                                       \n', stderr='2026-09-09T16:22:41.837887Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37\n2026-09-09T16:22:41.838263Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37\n2026-09-09T16:22:41.838506Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37\n2026-09-09T16:22:41.838691Z [info     ] setup plugin alembic.a

In [8]:
# Print the dependency graph straight from the parsed DAG - this is the evidence
# that the stages are wired in the right order.
import importlib.util
spec = importlib.util.spec_from_file_location('shopsense_dag', DAGS_DIR / 'shopsense_pipeline.py')
mod  = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
dag  = mod.dag

print(f'dag_id: {dag.dag_id}   schedule: {dag.schedule}   tasks: {len(dag.tasks)}\n')
for t in dag.topological_sort():
    downstream = sorted(t.downstream_task_ids)
    print(f'  {t.task_id:22} -> {", ".join(downstream) if downstream else "(end)"}')

dag_id: shopsense_pipeline   schedule: @daily   tasks: 7

  ingest_orders          -> quality_gate_bronze
  quality_gate_bronze    -> build_silver
  build_silver           -> quality_gate_silver
  quality_gate_silver    -> build_gold, refresh_rag_index
  build_gold             -> pipeline_complete
  refresh_rag_index      -> pipeline_complete
  pipeline_complete      -> (end)


## 5. Run 1 - the happy path

`airflow dags test` executes a real DAG run against the metadata database, task by task, honouring
the dependencies. Expect every task to end `success`.

In [9]:
os.environ.pop('SHOPSENSE_LANDING_OVERRIDE', None)     # use the clean landing file

# start the lineage log for this notebook from empty so the two runs are easy to read
lineage_log = PROJECT / 'reports' / 'lineage_events.jsonl'
if lineage_log.exists():
    lineage_log.unlink()

r1 = sh('airflow dags test shopsense_pipeline', quiet=True)
print('exit code:', r1.returncode, '(0 = the DAG run succeeded)')
for line in (r1.stdout + r1.stderr).splitlines():
    if any(k in line for k in ('ingested:', 'gate PASSED', 'silver:', 'gold:',
                               'rag index:', 'all stages completed', 'DagRun Finished')):
        print(' ', line.strip()[:190])

exit code: 0 (0 = the DAG run succeeded)
  ingested: {'source_file': 'orders_valid_20260908T194115Z.jsonl', 'rows': 347, 'distinct_order_id': 322}
  bronze gate PASSED: 8/8 expectations over 347 rows
  silver: {'created': True, 'rows': 322, 'inserted': 322, 'updated': 0}
  silver gate PASSED: 6/6 expectations over 322 rows
  gold: {'silver_rows': 322, 'gold_rows': 166, 'total_revenue_sar': 185917.0}
  rag index: {'documents': 12, 'chunks': 24}
  all stages completed and both quality gates passed
  2026-09-09T16:23:29.902715Z [info     ] DagRun Finished: dag_id=shopsense_pipeline, logical_date=2026-09-09 16:22:54.286588+00:00, run_id=manual__2026-09-09T16:22:59.982482+00:00, run_start_


In [10]:
from airflow.utils.session import create_session
from airflow.models import DagRun, TaskInstance
import pandas as pd

def run_states(dag_id='shopsense_pipeline'):
    with create_session() as s:
        dr = (s.query(DagRun).filter(DagRun.dag_id == dag_id)
                .order_by(DagRun.id.desc()).first())
        tis = s.query(TaskInstance).filter(TaskInstance.run_id == dr.run_id).all()
        rows = sorted([{'task_id': t.task_id, 'state': t.state} for t in tis],
                      key=lambda r: r['task_id'])
        return dr.run_id, dr.state, pd.DataFrame(rows)

run_id_1, state_1, states_1 = run_states()
print(f'DAG RUN 1  ->  {state_1}\n')
display(states_1)

DAG RUN 1  ->  success



,task_id,state
0,build_gold,success
1,build_silver,success
2,ingest_orders,success
3,pipeline_complete,success
4,quality_gate_bronze,success
5,quality_gate_silver,success
6,refresh_rag_index,success


## 6. Run 2 - the quality gate fails and the pipeline halts

Now point the ingestion task at the corrupted batch notebook 04 saved. Nothing else changes.
The bronze gate should fail, and **every task after it must not run**.

In [11]:
CORRUPT = PROJECT / 'data' / 'bronze_landing_corrupt.jsonl'
assert CORRUPT.exists(), 'run notebook 04 first - it writes the corrupted batch'
os.environ['SHOPSENSE_LANDING_OVERRIDE'] = str(CORRUPT)
print('DAG will ingest:', CORRUPT.name)

r2 = sh('airflow dags test shopsense_pipeline', quiet=True)
print('exit code:', r2.returncode, '(non-zero = the DAG run failed, which is what we want here)\n')
for line in (r2.stdout + r2.stderr).splitlines():
    if 'Quality gate FAILED' in line or 'expectations failed' in line or 'unexpected value' in line:
        print(' ', line.strip()[:190])

DAG will ingest: bronze_landing_corrupt.jsonl
exit code: 1 (non-zero = the DAG run failed, which is what we want here)

  src.quality.QualityGateFailed: Quality gate FAILED on bronze: 4/8 expectations failed
  - expect_column_values_to_not_be_null on customer_id: 1 unexpected value(s)
  - expect_column_values_to_be_between on quantity: 1 unexpected value(s)
  - expect_column_values_to_be_in_set on currency: 1 unexpected value(s)
  - expect_column_values_to_be_in_set on category: 1 unexpected value(s)


In [12]:
run_id_2, state_2, states_2 = run_states()
print(f'DAG RUN 2  ->  {state_2}\n')
display(states_2)

halted = states_2[states_2.state == 'upstream_failed']['task_id'].tolist()
print(f'\ntasks that never ran because the gate failed ({len(halted)}):')
for t in halted:
    print('  -', t)

DAG RUN 2  ->  failed



,task_id,state
0,build_gold,upstream_failed
1,build_silver,upstream_failed
2,ingest_orders,success
3,pipeline_complete,upstream_failed
4,quality_gate_bronze,failed
5,quality_gate_silver,upstream_failed
6,refresh_rag_index,upstream_failed



tasks that never ran because the gate failed (5):
  - build_gold
  - build_silver
  - pipeline_complete
  - quality_gate_silver
  - refresh_rag_index


In [13]:
# Side by side: the same DAG, two runs, one input difference.
comparison = (states_1.rename(columns={'state': 'run_1_clean_data'})
                      .merge(states_2.rename(columns={'state': 'run_2_corrupt_data'}),
                             on='task_id'))
display(comparison)

,task_id,run_1_clean_data,run_2_corrupt_data
0,build_gold,success,upstream_failed
1,build_silver,success,upstream_failed
2,ingest_orders,success,success
3,pipeline_complete,success,upstream_failed
4,quality_gate_bronze,success,failed
5,quality_gate_silver,success,upstream_failed
6,refresh_rag_index,success,upstream_failed


## 7. The lineage stream for both runs

In [14]:
from src import lineage as lin
events = lin.read_events(lineage_log)
print(f'{len(events)} OpenLineage events emitted across the two runs\n')
for e in events:
    marker = '  <-- gate failure' if e['eventType'] == 'FAIL' else ''
    print(f"  {e['eventType']:9} {e['job']['name']:22} {marker}")

18 OpenLineage events emitted across the two runs

  START     ingest_orders          
  COMPLETE  ingest_orders          
  START     quality_gate_bronze    
  COMPLETE  quality_gate_bronze    
  START     build_silver           
  COMPLETE  build_silver           
  START     quality_gate_silver    
  COMPLETE  quality_gate_silver    
  START     build_gold             
  COMPLETE  build_gold             
  START     refresh_rag_index      
  COMPLETE  refresh_rag_index      
  START     pipeline_complete      
  COMPLETE  pipeline_complete      
  START     ingest_orders          
  COMPLETE  ingest_orders          
  START     quality_gate_bronze    
  FAIL      quality_gate_bronze      <-- gate failure


In [15]:
import collections
counts = collections.Counter(e['eventType'] for e in events)
print('event types:', dict(counts))

fail = [e for e in events if e['eventType'] == 'FAIL']
if fail:
    print('\nFAIL event error facet:\n')
    print(fail[0]['run']['facets']['errorMessage']['message'][:500])

event types: {'START': 9, 'COMPLETE': 8, 'FAIL': 1}

FAIL event error facet:

Quality gate FAILED on bronze: 4/8 expectations failed
  - expect_column_values_to_not_be_null on customer_id: 1 unexpected value(s)
  - expect_column_values_to_be_between on quantity: 1 unexpected value(s)
  - expect_column_values_to_be_in_set on currency: 1 unexpected value(s)
  - expect_column_values_to_be_in_set on category: 1 unexpected value(s)


In [16]:
# The dataset-level lineage the events describe
edges = []
for e in events:
    for o in e.get('outputs', []):
        for i in e.get('inputs', []):
            edges.append((i['name'], e['job']['name'], o['name']))
for src_ds, job, out_ds in sorted(set(edges)):
    print(f'  {src_ds:34} --[{job}]-->  {out_ds}')

  kafka.orders.valid                 --[ingest_orders]-->  lakehouse_dag.bronze.orders
  lakehouse_dag.bronze.orders        --[build_silver]-->  lakehouse_dag.silver.orders
  lakehouse_dag.silver.orders        --[build_gold]-->  lakehouse_dag.gold.daily_category_revenue


## 8. Persist the DAG and write the stage report

In [17]:
import shutil, json, datetime

(PROJECT / 'dags').mkdir(exist_ok=True)
shutil.copy2(DAGS_DIR / 'shopsense_pipeline.py', PROJECT / 'dags' / 'shopsense_pipeline.py')
print('DAG saved to', PROJECT / 'dags' / 'shopsense_pipeline.py')

run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
report = {
    'run_id': run_id,
    'stage': 'orchestration',
    'engine': f'apache-airflow {airflow.__version__} (python {PY})',
    'dag_id': dag.dag_id,
    'schedule': str(dag.schedule),
    'tasks': [t.task_id for t in dag.topological_sort()],
    'dependencies': {t.task_id: sorted(t.downstream_task_ids) for t in dag.tasks},
    'run_1_clean': {'dag_run_state': str(state_1),
                    'task_states': states_1.set_index('task_id')['state'].astype(str).to_dict()},
    'run_2_corrupt': {'dag_run_state': str(state_2),
                      'task_states': states_2.set_index('task_id')['state'].astype(str).to_dict(),
                      'halted_by_gate': halted},
    'lineage_events': dict(counts),
    'finished_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
out = PROJECT / 'reports' / f'orchestration_report_{run_id}.json'
out.write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))

DAG saved to /content/drive/MyDrive/sdaia_capstone/dags/shopsense_pipeline.py
{
  "run_id": "20260909T162405Z",
  "stage": "orchestration",
  "engine": "apache-airflow 3.1.8 (python 3.13)",
  "dag_id": "shopsense_pipeline",
  "schedule": "@daily",
  "tasks": [
    "ingest_orders",
    "quality_gate_bronze",
    "build_silver",
    "quality_gate_silver",
    "build_gold",
    "refresh_rag_index",
    "pipeline_complete"
  ],
  "dependencies": {
    "ingest_orders": [
      "quality_gate_bronze"
    ],
    "quality_gate_bronze": [
      "build_silver"
    ],
    "build_silver": [
      "quality_gate_silver"
    ],
    "quality_gate_silver": [
      "build_gold",
      "refresh_rag_index"
    ],
    "build_gold": [
      "pipeline_complete"
    ],
    "refresh_rag_index": [
      "pipeline_complete"
    ],
    "pipeline_complete": []
  },
  "run_1_clean": {
    "dag_run_state": "success",
    "task_states": {
      "build_gold": "success",
      "build_silver": "success",
      "ingest_or

## The capstone is complete

| # | Deliverable | Pts | Evidence |
|---|---|---|---|
| 1 | Ingestion | 20 | `01_ingestion_kafka.ipynb` |
| 2 | Delta Lakehouse | 25 | `02_delta_lakehouse.ipynb` |
| 3 | RAG Pipeline | 25 | `03_rag_pipeline.ipynb` |
| 4 | Orchestration | 15 | `05_airflow_orchestration.ipynb` + `dags/shopsense_pipeline.py` |
| 5 | Quality Gate + Lineage | 15 | `04_quality_gate_lineage.ipynb` + `src/quality.py`, `src/lineage.py` |
| | **Total** | **100** | |

`File -> Save`, then push this notebook, `src/` and `dags/` to GitHub.